# RAG Data Ingestion - Inspection Notebook

This notebook is a **read-only inspection / example-retrieval notebook**. It no
longer builds or rebuilds the production Chroma index in place.

**Canonical RAG rebuild** (safe, staged, validated - builds into a throwaway
staging directory, validates the result, and only then swaps it in for
`chroma_db/`; a failure at any point leaves the bundled index untouched):

```
REBUILD_RAG.bat                 # Windows one-click
python -m tools.rebuild_rag     # direct CLI
python -m tools.rebuild_rag --validate-only   # offline check, no API key
```

See `tools/rebuild_rag.py` for the implementation - this notebook imports its
source-discovery/chunking constants and functions directly, so the two can
never silently drift apart.

What this notebook still does:
1. Loads `GEMINI_API_KEY` from `.env` (needed for the embedding calls below).
2. Inspects the active source corpus and chunk plan via
   `tools.rebuild_rag.build_chunk_plan()` - offline, no embedding/network call.
3. Opens the **existing** `chroma_db/` read-only and runs one example
   similarity search against it.

It never deletes a collection, never writes into `chroma_db/`, and never
duplicates the staged-build/swap logic.

## 1) SETUP - load API key from .env
Same mechanism as in `agent.ipynb` / `app.py`: `GEMINI_API_KEY` is loaded via
`python-dotenv` and validated before use.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Repository-root resolution: this notebook can be launched with either the
# repo root or notebooks/ as the working directory. Walk up at most one
# level to find the directory that actually contains "Rag Database".
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "Rag Database").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env!")
os.environ["GOOGLE_API_KEY"] = api_key  # for langchain-google-genai

import tools.rebuild_rag as rebuild_rag

print("Setup complete. GEMINI_API_KEY loaded; tools.rebuild_rag importable.")

## 2) SOURCE DISCOVERY & CHUNK PLAN (offline, read-only)
Reuses `tools.rebuild_rag.build_chunk_plan()` - the exact same source
discovery / loading / chunking code the canonical rebuild uses - so this
inspection can never silently drift from what a real rebuild would do. No
embedding call or network access happens in this cell.

Only `Rag Database/box1_patterns/` (box 1) and `Rag Database/box2_domain/`
(box 2) are scanned; `Rag Database/raw_source_archive/` is never indexed.

In [ ]:
plan = rebuild_rag.build_chunk_plan()

print(f"Active sources: {plan.source_count}")
print(f"Total chunks:   {plan.chunk_count}")
print(f"Box counts:     {plan.box_counts}")

mismatches = rebuild_rag.verify_expected_corpus(plan)
if mismatches:
    print("
NOTE: corpus differs from the last accepted baseline:")
    for m in mismatches:
        print(f"  - {m}")
else:
    print("
Matches the accepted baseline (11 sources / 503 chunks / box1=440, box2=63).")

## 3) INSPECT THE EXISTING INDEX (read-only, no rebuild)
Opens the **already-built** `chroma_db/` for inspection only - this cell does
not create, delete, or modify the index. If `chroma_db/` is missing or empty,
run the canonical rebuild first (`REBUILD_RAG.bat` or
`python -m tools.rebuild_rag`).

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model=rebuild_rag.EMBEDDING_MODEL)
CHROMA_DIR = str(REPO_ROOT / "chroma_db")

vectorstore = Chroma(
    collection_name=rebuild_rag.COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
print(f"Indexed vectors in {CHROMA_DIR}: {vectorstore._collection.count()}")

## 4) EXAMPLE RETRIEVAL - similarity search (Microservices)
An architecture-related test query against the existing vector store. The
top-3 hits are printed together with their source (metadata).

In [ ]:
query = "What are the benefits of a microservices architecture?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}
")
for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "?")
    print(f"--- Result {i} | Source: {source} | Page: {page} ---")
    print(doc.page_content[:300].strip())
    print()